In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

In [ ]:
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.compose import ColumnTransformer

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/sanjanbm/modified-titanic-data/train.csv')
df.sample(3)

In [ ]:
df.drop(columns = ['PassengerId','Name','Ticket','Cabin'], inplace = True)
df.sample(4)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['Survived']),
                                                 df['Survived'],
                                                 test_size=0.2,
                                                random_state=42)

X_train.shape, X_test.shape

In [ ]:
X_train.head()

In [ ]:
y_train.head()

## Apply column transformer

In [ ]:
# Imputation transformer

tf1 = ColumnTransformer([
    ('impute_age', SimpleImputer(), [2]),
    ('impute_embarked', SimpleImputer(strategy='most_frequent'), [6]), 
], remainder='passthrough')

tf1

In [ ]:
# One hot encoding

tf2 = ColumnTransformer([
    ('ohe_sex_embarked', OneHotEncoder(handle_unknown='ignore', sparse_output=False), [1, 6])
], remainder='passthrough')

In [ ]:
# scaling

tf3 = ColumnTransformer([
    ('scale', MinMaxScaler(), slice(0, 10))
])

In [ ]:
# Feature selection

tf4 = SelectKBest(score_func=chi2, k=5)

In [ ]:
# train the model

tf5 = DecisionTreeClassifier()

## Create Pipeline

In [ ]:
pipe = Pipeline([
    ('trf1', tf1),
    ('trf2', tf2),
    ('trf3', tf3),
    ('trf4', tf4),
    ('trf5', tf5),
])

### Pipeline Vs make_pipeline
Pipeline requires naming of steps, make_pipeline does not.

(Same applies to ColumnTransformer vs make_column_transformer)

In [ ]:
# Alternate Syntax
pipe = make_pipeline(tf1,tf2,tf3,tf4,tf5)

In [ ]:
pipe.fit(X_train,y_train)

## Explore the Pipeline

In [ ]:
pipe.named_steps

In [ ]:
# Display Pipeline

from sklearn import set_config
set_config(display='diagram')

In [ ]:
y_pred = pipe.predict(X_test)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import accuracy_score
print(f"{accuracy_score(y_test,y_pred)*100}%")

## Cross Validation using Pipeline

In [ ]:
from sklearn.model_selection import cross_val_score

cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy').mean()

## GridSearch using Pipeline

In [ ]:
pipe = Pipeline([
    ('imputer', tf1),
    ('encoder', tf2),
    ('scaler', tf3),
    ('feature_selection', tf4),
    ('model', DecisionTreeClassifier()) # Let's call it 'model'
])

# Then your params grid is much cleaner:
params = {
    'model__max_depth': [3, 5, 10],
    'feature_selection__k': [2, 5, 8]
}

In [ ]:
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(pipe, params, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

In [ ]:
grid.best_score_

In [ ]:
grid.best_params_

In [ ]:
# export 
import pickle
pickle.dump(pipe,open('pipe.pkl','wb'))